# Week 4: Files, URLs and `pandas`

**GEOG0111 Scientific Computing**
MSc Environmental Modelling · MSc Remote Sensing and Environmental Mapping · UCL Geography

Every analysis starts with data that live **somewhere**: in a folder, on a web server or behind an API. This week covers:

1. **Files and folders** with `pathlib`: paths, creating, finding (`glob`), copying and deleting files;
2. **dates and times** with `datetime`;
3. **structured text** formats: JSON and YAML;
4. **downloading data from URLs**, safely and reproducibly;
5. **`pandas`** for tables: reading, inspecting, selecting, filtering, calculating, grouping and saving.

The real datasets this week are the two text datasets behind the Part B suggested topics: the **Met Office Heathrow station record** (urban heat island topic) and the **NOAA Oceanic Niño Index** (Amazon fire topic). By the end you will have a **reusable, function-based pipeline** that downloads, cleans, summarises and saves a climate record, which is exactly the kind of workflow Part B expects.

> **SOLUTIONS VERSION: for the instructor.** Release after the session. Suggested answers are in the "Your code here" cells.

## Learning outcomes

By the end of this notebook you should be able to:

- build and inspect file paths with `pathlib.Path`, and create, find, copy and delete files safely;
- read and write text files with `Path.read_text()`/`write_text()` and `with open(...)`;
- parse and format dates with `datetime`, including day of year;
- save and load metadata and settings as JSON and YAML;
- download a file from a URL, check that it worked, and handle failures with `try`/`except`;
- keep credentials out of notebooks;
- load tables into a `pandas` DataFrame, inspect them, and select, filter, sort, group and save data;
- read a real, messy whitespace-separated data file, and clean flags and missing values.

### How to use this notebook

- Run the cells **in order**. Several exercises build on files created earlier.
- The module's AI rules apply. Good prompts this week: *"Explain this pandas error in simple terms"*, *"What does this read_csv argument do?"* or *"My DataFrame has columns X, Y, Z: suggest how to filter rows where..."*. **Always check column names and units** in AI-generated pandas code.
- **Working locally in VS Code?** This notebook uses no shell commands, so it works on Windows, macOS and Linux. You need `pandas`, `requests` and `pyyaml` (`%pip install pandas requests pyyaml`).

In [ ]:
# Imports for this notebook: standard library first, then third-party packages
from pathlib import Path
from datetime import datetime, timedelta
import shutil
import json

import pandas as pd
import requests
import yaml

print('pandas', pd.__version__)

---
# 1. Files and folders with `pathlib`

## 1.1 Kinds of data you will meet

| Kind | Examples | Python tools |
|---|---|---|
| Plain text | station records, logs, notes | `Path.read_text()`, `open()` |
| Tables (CSV, TXT, Excel) | climate series, scene lists, field data | `pandas` (this week) |
| JSON / YAML | metadata, API responses, configuration | `json`, `yaml` (this week) |
| Raster | GeoTIFF, netCDF, satellite images | `rasterio`, `xarray` (Weeks 6–7) |
| Vector | boundaries, points, polygons | `geopandas` (Week 7) |
| Cloud catalogues | Earth Engine collections, STAC | `ee`, `geemap` (Week 8) |

Files are either **text** (human-readable characters, such as CSV, JSON or `.txt`) or **binary** (raw bytes, such as GeoTIFF, HDF, netCDF, PNG or ZIP). Text is read with `read_text()`, and binary with `read_bytes()`.

## 1.2 Paths

A **path** says where a file is: `/content/data/heathrow.txt`. The modern way to handle paths is the `Path` class from `pathlib`. Its `/` operator joins parts correctly on **every** operating system (Windows uses `\`, macOS and Linux use `/`), so never build paths by gluing strings together.

In [ ]:
cwd = Path.cwd()                  # where the notebook is running (in Colab: /content)
print('working directory:', cwd)

data_dir = Path('data')           # a RELATIVE path (relative to cwd)
raw_dir = data_dir / 'raw'        # join with /
print(raw_dir, '->', raw_dir.resolve())   # resolve() gives the ABSOLUTE path

A `Path` knows its own parts:

| Attribute / method | Example for `Path('data/S2/2024/ndvi_2024-07.tif')` |
|---|---|
| `.name` | `'ndvi_2024-07.tif'` |
| `.stem` | `'ndvi_2024-07'` |
| `.suffix` | `'.tif'` |
| `.parent` | `Path('data/S2/2024')` |
| `.parts` | `('data', 'S2', '2024', 'ndvi_2024-07.tif')` |
| `.with_suffix('.png')` | `Path('data/S2/2024/ndvi_2024-07.png')` |
| `.exists()`, `.is_file()`, `.is_dir()` | `True` / `False` |
| `.mkdir(parents=True, exist_ok=True)` | create the folder (and any missing parents) |
| `.iterdir()` | the contents of a folder |
| `.glob('*.tif')` | find files matching a pattern |
| `.stat().st_size` | file size in bytes |

In [ ]:
# Create the folders we will use this week. exist_ok=True: no error if they already exist
for folder in [data_dir / 'raw', data_dir / 'processed', data_dir / 'images']:
    folder.mkdir(parents=True, exist_ok=True)

for item in sorted(data_dir.iterdir()):
    print(item, '| is a folder:', item.is_dir())

#### Exercise 1

For `p = Path('data/S2/2024/ndvi_2024-07.tif')`:

1. Print its name, stem, suffix, parent and parts.
2. Create the path of a figure with the same stem but a `.png` suffix, inside `data/images` (hint: `.name`, `.with_suffix()` and `/`).
3. Does `p` exist? Why not?
4. From the **stem**, extract the year and the month as integers (Week 2 string skills).

In [ ]:
p = Path('data/S2/2024/ndvi_2024-07.tif')
print(p.name, p.stem, p.suffix, p.parent, p.parts)

fig_path = data_dir / 'images' / p.with_suffix('.png').name
print(fig_path)

print('exists?', p.exists())   # False: a Path is only an address; we never created the file

date_part = p.stem.split('_')[1]          # '2024-07'
year, month = int(date_part[:4]), int(date_part[5:7])
print(year, month)

## 1.3 Writing and reading text files

For small files, `Path.write_text()` and `Path.read_text()` do everything in one line. `write_text()` **overwrites** an existing file.

In [ ]:
notes = data_dir / 'raw' / 'field_notes.txt'

notes.write_text(
    'Site: Regent\'s Park, London\n'
    'Date: 2024-07-15\n'
    'Notes: dry grass, scattered trees, Sentinel-2 overpass at 11:06 UTC\n'
)

text = notes.read_text()
print(text)
print('number of lines:', len(text.splitlines()))

### `with open(...)`: the general way

`open()` gives finer control, for example writing a file line by line inside a loop, or reading a large file one line at a time without loading it all into memory. The `with` block **closes the file automatically** when it ends, even if an error occurs.

| Mode | Meaning |
|---|---|
| `'r'` | read (the default) |
| `'w'` | write: **overwrites** the file |
| `'a'` | append to the end |
| `'rb'` / `'wb'` | read / write binary |

In [ ]:
log_file = data_dir / 'processed' / 'processing_log.txt'

# write line by line in a loop
with open(log_file, 'w') as f:
    for month in range(1, 4):
        f.write(f'2024-{month:02d}: NDVI composite created\n')

# append one more line
with open(log_file, 'a') as f:
    f.write('2024-04: skipped (too cloudy)\n')

# read one line at a time
with open(log_file) as f:
    for line in f:
        print(line.rstrip())      # rstrip() removes the trailing newline

#### Exercise 2

1. Create `data/raw/scene_notes.txt` with **three** lines: a satellite name, an acquisition date and a short observation. Read it back and print it.
2. Using `with open(..., 'a')`, append a fourth line: `Checked by: <your initials>`.
3. Read the file with `with open(...)` and a loop, and print each line **with its line number** (hint: `enumerate(f, start=1)`).

In [ ]:
scene_notes = data_dir / 'raw' / 'scene_notes.txt'
scene_notes.write_text('Satellite: Sentinel-2B\nDate: 2024-07-15\nObservation: clear sky over central London\n')
print(scene_notes.read_text())

with open(scene_notes, 'a') as f:
    f.write('Checked by: MM\n')

with open(scene_notes) as f:
    for i, line in enumerate(f, start=1):
        print(i, line.rstrip())

## 1.4 File information and finding files with `glob`

`.stat()` gives information about a file, such as its size and modification time. `glob()` finds files whose names match a **pattern**:

| Pattern | Matches |
|---|---|
| `*` | any characters (including none) |
| `?` | exactly one character |
| `[0-9]` | one character from a range |
| `**` | any number of sub-folders (use `rglob()` or `glob('**/...')`) |

To practise, let's create twelve empty "monthly NDVI images". `touch()` creates an empty file.

In [ ]:
img_dir = data_dir / 'images'
for month in range(1, 13):
    (img_dir / f'ndvi_2024-{month:02d}.tif').touch()

all_tifs = sorted(img_dir.glob('*.tif'))          # glob gives no guaranteed order: sort!
print(len(all_tifs), 'files, first:', all_tifs[0].name, 'last:', all_tifs[-1].name)

q1 = sorted(img_dir.glob('ndvi_2024-0[1-3].tif'))
print('Q1:', [f.name for f in q1])

info = notes.stat()
print(notes.name, info.st_size, 'bytes, modified', datetime.fromtimestamp(info.st_mtime))

> Notice how the **zero-padding** from Week 2 makes `sorted()` put the months in the right order.

#### Exercise 3

1. Use `glob` to list only the **summer** images (June to August), sorted.
2. Use `rglob('*.txt')` on `data_dir` to find **all** text files in all sub-folders, and print each one's path and size in bytes.
3. Use a list comprehension to make a list of the **months** (as integers) of all the `.tif` files, extracted from their stems.

In [ ]:
summer = sorted(img_dir.glob('ndvi_2024-0[6-8].tif'))
print([f.name for f in summer])

for f in sorted(data_dir.rglob('*.txt')):
    print(f, f.stat().st_size, 'bytes')

months = [int(f.stem[-2:]) for f in sorted(img_dir.glob('*.tif'))]
print(months)

## 1.5 Copying, renaming and deleting, carefully

| Task | Tool |
|---|---|
| copy a file | `shutil.copy(src, dst)` |
| copy a folder | `shutil.copytree(src, dst)` |
| rename / move | `path.rename(new_path)` |
| delete a file | `path.unlink(missing_ok=True)` |
| delete an **empty** folder | `path.rmdir()` |
| delete a folder **and everything in it** | `shutil.rmtree(path)` |

> **Deleting is permanent: there is no recycle bin.** Always print the path you are about to delete first, and never delete with a pattern you have not checked.

In [ ]:
backup_dir = data_dir / 'backup'
backup_dir.mkdir(exist_ok=True)

copied = shutil.copy(notes, backup_dir / notes.name)
print('copied to', copied)

renamed = Path(copied).rename(backup_dir / 'field_notes_2024-07-15.txt')
print('renamed to', renamed)

# delete safely: show what will be deleted first
print('about to delete:', renamed.resolve())
renamed.unlink(missing_ok=True)
print('still exists?', renamed.exists())

---
# 2. Dates and times with `datetime`

Dates are everywhere in EO and environmental data: acquisition times, time series and revisit cycles. The `datetime` module turns date **strings** into date **objects** that you can compare, sort and do arithmetic with.

| Task | Code |
|---|---|
| string → datetime | `datetime.strptime('20240715T110619', '%Y%m%dT%H%M%S')` |
| datetime → string | `dt.strftime('%Y-%m-%d')` |
| parts | `dt.year`, `dt.month`, `dt.day`, `dt.hour` |
| day of year | `dt.timetuple().tm_yday` or `dt.strftime('%j')` |
| add or subtract time | `dt + timedelta(days=5)` |
| difference | `dt2 - dt1` gives a `timedelta` |

Common format codes: `%Y` year, `%m` month, `%d` day, `%H` hour, `%M` minute, `%S` second, `%j` day of year, `%b` short month name.

In [ ]:
sensing = datetime.strptime('20240715T110619', '%Y%m%dT%H%M%S')
print(sensing, type(sensing))
print(sensing.strftime('%d %b %Y, %H:%M UTC'))
print('day of year:', sensing.timetuple().tm_yday)

next_pass = sensing + timedelta(days=5)          # Sentinel-2 revisit with two satellites
print('next revisit:', next_pass.date())

launch = datetime(2017, 3, 7)                    # Sentinel-2B launch date
print('days since launch:', (sensing - launch).days)

#### Exercise 4

1. From the product name `S2A_MSIL2A_20240620T105621_N0510_R094_T30UXC_20240620T150010.SAFE`, extract the sensing time (the third field) and convert it to a `datetime`.
2. Print it as `2024-06-20 (DOY 172)`.
3. Using a loop and `timedelta`, print the dates of the next **six** Sentinel-2A passes. A single satellite revisits every **10 days**.
4. MODIS file names encode dates as `AYYYYDDD`, e.g. `A2024197`. Convert `'A2024197'` to a normal date. (Hint: the format code `%j`.)

In [ ]:
name = 'S2A_MSIL2A_20240620T105621_N0510_R094_T30UXC_20240620T150010.SAFE'
t = datetime.strptime(name.split('_')[2], '%Y%m%dT%H%M%S')
print(f"{t:%Y-%m-%d} (DOY {t.timetuple().tm_yday})")

for k in range(1, 7):
    print((t + timedelta(days=10 * k)).date())

modis_date = datetime.strptime('A2024197'[1:], '%Y%j')
print(modis_date.date())     # 2024-07-15

---
# 3. Structured text: JSON and YAML

Both formats store **dictionaries and lists** as text, so you can save Python structures and load them back later, in another notebook or another language.

- **JSON** is everywhere: web APIs, STAC catalogues, Earth Engine metadata, GeoJSON.
- **YAML** is easier for people to read and write, so it is popular for **configuration files** (e.g. which area, dates and thresholds an analysis should use).

In [ ]:
scene_meta = {
    'mission': 'S2B',
    'level': 'L2A',
    'date': '2024-07-15',
    'tile': '30UXC',
    'cloud_cover': 12.5,
    'bands': ['B02', 'B03', 'B04', 'B08'],
    'created': datetime.now().isoformat(timespec='seconds'),
}

meta_file = data_dir / 'processed' / 'scene_meta.json'
meta_file.write_text(json.dumps(scene_meta, indent=2))   # dict -> JSON text
print(meta_file.read_text())

loaded = json.loads(meta_file.read_text())               # JSON text -> dict
print(type(loaded), loaded['bands'])

In [ ]:
config_text = '''
# Analysis settings for a London NDVI study
study_area: Greater London
bbox: [-0.51, 51.28, 0.33, 51.69]   # min_lon, min_lat, max_lon, max_lat
years: [2018, 2024]
months: [6, 7, 8]
max_cloud: 20
indices:
  - NDVI
  - NDBI
'''
config_file = data_dir / 'config.yaml'
config_file.write_text(config_text)

config = yaml.safe_load(config_file.read_text())     # always use safe_load
print(config)
print('summer months:', config['months'], '| cloud threshold:', config['max_cloud'])

> Keeping settings in a config file (rather than scattered through the code) means you can rerun an analysis for a different city or year by editing **one** file. This makes your work more reproducible.

#### Exercise 5

1. Change `config` in Python: set `study_area` to `'Greater Manchester'` and `max_cloud` to `10`. Save it as `data/config_manchester.yaml` with `yaml.safe_dump(config, sort_keys=False)`, and print the file.
2. Make a **list of two dictionaries**, each describing one scene (mission, date, cloud cover). Save it as JSON, load it back, and use `assert` to check that the loaded list equals the original.

In [ ]:
config['study_area'] = 'Greater Manchester'
config['max_cloud'] = 10
manc = data_dir / 'config_manchester.yaml'
manc.write_text(yaml.safe_dump(config, sort_keys=False))
print(manc.read_text())

scenes = [
    {'mission': 'S2A', 'date': '2024-06-20', 'cloud': 8.1},
    {'mission': 'S2B', 'date': '2024-07-15', 'cloud': 12.5},
]
scenes_file = data_dir / 'processed' / 'scenes.json'
scenes_file.write_text(json.dumps(scenes, indent=2))
assert json.loads(scenes_file.read_text()) == scenes
print('round trip OK')

---
# 4. Getting data from URLs

A **URL** is the address of a resource on the web. There are two common approaches:

1. **Download** the file, then read the local copy. This is best for reproducibility: you know exactly which version you analysed, and you don't download it again every time you run the notebook.
2. **Read directly** from the URL, e.g. `pd.read_csv(url)`. This is quick for small, stable files.

## 4.1 Handling errors: `try` / `except`

Downloads fail: the network drops, a server is down, a URL changes. Instead of crashing, code can **catch** the error and react:

```python
try:
    risky_operation()
except SomeError as e:
    print('it failed:', e)
```

In [ ]:
try:
    value = float('23.4#')
except ValueError as e:
    print('Could not convert:', e)
    value = float('23.4#'.strip('#'))
print(value)

## 4.2 A download function

We use the `requests` package, which is pre-installed in Colab. The function below follows the Part B rules: everything it needs comes in as an argument, it **returns** a value, and it has a docstring.

- `requests.get(url, timeout=60)` fetches the URL; give up after 60 s.
- `r.raise_for_status()` turns an HTTP error (e.g. **404** Not Found, **403** Forbidden) into a Python error. **200** means OK.
- `r.content` holds the raw bytes, which works for both text and binary files.

In [ ]:
def download_file(url, out_path, overwrite=False, timeout=60):
    """Download a file from a URL to a local path.

    Parameters
    ----------
    url : str
        Address of the file.
    out_path : str or Path
        Where to save it. Parent folders are created if needed.
    overwrite : bool, optional
        If False (default) and the file already exists and is not empty,
        it is not downloaded again.
    timeout : int, optional
        Seconds to wait for the server (default 60).

    Returns
    -------
    Path
        Path of the local file.
    """
    out_path = Path(out_path)
    if out_path.exists() and out_path.stat().st_size > 0 and not overwrite:
        print(f'{out_path} already exists, not downloading again')
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()                      # stop with an error if not 200 OK
    out_path.write_bytes(r.content)
    print(f'downloaded {out_path} ({out_path.stat().st_size:,} bytes)')
    return out_path

Now download the Heathrow record. The `try`/`except` means that if the download fails (for example on a laptop without internet access), the notebook falls back to a **short excerpt of real lines** from the file, so everything below still runs.

In [ ]:
HEATHROW_URL = 'https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/heathrowdata.txt'
heathrow_file = data_dir / 'raw' / 'heathrowdata.txt'

# A few real lines from the file, used ONLY if the download fails
HEATHROW_EXCERPT = """Heathrow (London Airport)
Location 507800E 176700N, Lat 51.479 Lon -0.449, 25m amsl
Estimated data is marked with a * after the value.
Missing data (more than 2 days missing in month) is marked by  ---.
Sunshine data taken from an automatic Kipp & Zonen sensor marked with a #, otherwise sunshine data taken from a Campbell Stokes recorder.
   yyyy  mm   tmax    tmin      af    rain     sun
              degC    degC    days      mm   hours
   1948   1    8.9     3.3    ---     85.0    ---
   1948   2    7.9     2.2    ---     26.0    ---
   1948   3   14.2     3.8    ---     14.0    ---
   2026   5   21.9    10.8       0     4.2   206.3#  Provisional
   2026   6   25.1    14.9       0    81.6   207.4#  Provisional
   2026   7   29.3    16.2       0     0.0   327.3#  Provisional
   2026   8   27.5    15.7       0    12.0   257.2#  Provisional
   2026   9   23.6    13.8       0    26.2   147.8#  Provisional
"""

try:
    download_file(HEATHROW_URL, heathrow_file)
except Exception as e:
    print('Download failed:', e)
    print('-> using the short offline excerpt instead (results below will be limited)')
    print('   Once you are online, rerun with download_file(HEATHROW_URL, heathrow_file, overwrite=True)')
    heathrow_file.write_text(HEATHROW_EXCERPT)

In [ ]:
# Look before you read: always inspect a new file first
lines = heathrow_file.read_text().splitlines()
print(len(lines), 'lines\n')
print('\n'.join(lines[:10]))
print('...')
print('\n'.join(lines[-3:]))

#### Exercise 6

The **Oceanic Niño Index (ONI)** from NOAA measures El Niño (positive anomalies) and La Niña (negative anomalies). It is the ENSO index used in the Amazon fire topic for Part B.

URL: `https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt`

1. Use `download_file()` to save it as `data/raw/oni.ascii.txt`. Wrap the call in `try`/`except`, and if it fails write the excerpt given in the cell below instead.
2. Print the number of lines, the header line and the last 3 lines. How are the columns separated? What do you think `SEAS` means?
3. Call `download_file()` again. What happens, and why is that useful?

In [ ]:
ONI_URL = 'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt'
# A few real lines from the file, used ONLY if the download fails
ONI_EXCERPT = """ SEAS  YR   TOTAL   ANOM
  DJF 1950  25.01  -1.32
  JFM 1950  25.36  -1.20
  FMA 1950  25.88  -1.12
  MAM 1950  26.24  -1.08
  MJJ 2026  29.02   1.39
  JJA 2026  29.09   1.80
  JAS 2026  29.12   2.16
"""

In [ ]:
oni_file = data_dir / 'raw' / 'oni.ascii.txt'
try:
    download_file(ONI_URL, oni_file)
except Exception as e:
    print('Download failed:', e, '-> using excerpt')
    oni_file.write_text(ONI_EXCERPT)

oni_lines = oni_file.read_text().splitlines()
print(len(oni_lines), 'lines')
print(oni_lines[0])
print('\n'.join(oni_lines[-3:]))
# Columns are separated by a variable number of spaces (whitespace), not commas.
# SEAS = a 3-month 'season' of overlapping months, e.g. DJF = Dec-Jan-Feb.

download_file(ONI_URL, oni_file)
# Second call: the file exists, so it is not downloaded again. This saves time and
# server load, and keeps the analysis tied to the same version of the data.

## 4.3 Data provenance

Record **where** and **when** you got your data. Datasets get updated (the Heathrow file changes every month), so your results depend on the download date. A small JSON "provenance" file next to the data is enough:

In [ ]:
provenance = {
    'dataset': 'Met Office historic station data: Heathrow',
    'url': HEATHROW_URL,
    'downloaded': datetime.fromtimestamp(heathrow_file.stat().st_mtime).isoformat(timespec='seconds'),
    'file': str(heathrow_file),
    'size_bytes': heathrow_file.stat().st_size,
}
(data_dir / 'raw' / 'heathrow_provenance.json').write_text(json.dumps(provenance, indent=2))
print(json.dumps(provenance, indent=2))

## 4.4 Data that need a login: keep credentials out of notebooks

Many EO archives need an account: for example **NASA Earthdata** (MODIS, Landsat, Harmonized Landsat-Sentinel), the **Copernicus Data Space Ecosystem** (Sentinel), and **Google Earth Engine** (Week 8).

**Never type a username, password or API key into a notebook cell.** Notebooks get shared and submitted, and Part B asks you to keep everything anonymous. Instead:

- **Colab Secrets:** click the 🔑 icon in Colab's left sidebar, store e.g. `EARTHDATA_PASSWORD`, then read it in code with `from google.colab import userdata; userdata.get('EARTHDATA_PASSWORD')`.
- **`getpass`:** asks for the value when the cell runs, without displaying it: `from getpass import getpass; pw = getpass('Password: ')`.
- **Package login helpers:** e.g. `earthaccess.login()` for NASA data, or `ee.Authenticate()` for Earth Engine, which store a token for you.

We don't download any password-protected data this week. In Week 7–8 we use Earth Engine, which handles access for you.

---
# 5. Tables with `pandas`

`pandas` is the standard Python package for tabular data. Its central object is the **DataFrame**: a table with **named columns** (variables) and an **index** (row labels). Each column is a **Series**.

## 5.1 Creating and inspecting a DataFrame

A DataFrame can be built from a dictionary of lists. Here is the scene catalogue from Week 3, now as a table:

In [ ]:
scenes = pd.DataFrame({
    'date':    ['2024-06-05', '2024-06-20', '2024-07-15', '2024-07-30', '2024-08-14', '2024-09-09'],
    'mission': ['S2B', 'S2A', 'S2B', 'S2A', 'S2B', 'S2A'],
    'cloud':   [64.2, 8.1, 12.5, 47.9, 2.3, 18.4],
    'red':     [0.081, 0.052, 0.058, 0.074, 0.061, 0.066],   # mean red reflectance over a park
    'nir':     [0.302, 0.401, 0.388, 0.331, 0.352, 0.341],   # mean NIR reflectance
})
scenes

*(Illustrative values, made up for practice.)*

**Always inspect a DataFrame first:**

| Method | Tells you |
|---|---|
| `df.head(n)` / `df.tail(n)` | first / last `n` rows |
| `df.shape` | (rows, columns) |
| `df.columns` | column names |
| `df.dtypes` / `df.info()` | data type of each column, and missing values |
| `df.describe()` | summary statistics of the numeric columns |
| `df['col'].unique()`, `df['col'].value_counts()` | distinct values and their counts |
| `df.isna().sum()` | number of missing values per column |

In [ ]:
print(scenes.shape)
print(list(scenes.columns))
scenes.info()

In [ ]:
scenes.describe()

In [ ]:
scenes['mission'].value_counts()

## 5.2 Selecting columns and rows

| You want | Code |
|---|---|
| one column (a Series) | `df['cloud']` |
| several columns (a DataFrame) | `df[['date', 'cloud']]` (note the double brackets) |
| rows by **condition** | `df[df['cloud'] < 20]` |
| several conditions | `df[(df['cloud'] < 20) & (df['mission'] == 'S2B')]` |
| rows and columns by **label** | `df.loc[row_labels, column_names]` |
| rows and columns by **position** | `df.iloc[0:3, 0:2]` |

> **Combining conditions:** use `&` (and), `|` (or) and `~` (not), and put **each condition in brackets**. Python's `and`/`or` do not work on whole columns; they give `ValueError: The truth value of a Series is ambiguous`.

In [ ]:
print(scenes['cloud'].mean())

clear = scenes[scenes['cloud'] < 20]
clear

In [ ]:
scenes.loc[scenes['mission'] == 'S2A', ['date', 'cloud']]

#### Exercise 7

1. Select only the `date` and `mission` columns.
2. Select the scenes that are **clear (< 20 %) and from S2B**.
3. Select the scenes that are **either** very clear (< 5 %) **or** very cloudy (> 60 %).
4. Try writing condition 2 with `and` instead of `&`. Read the error, then comment the line out.
5. How many scenes have cloud cover below 20 %? (Hint: `len()`, or `.sum()` on the True/False condition.)

In [ ]:
print(scenes[['date', 'mission']])

print(scenes[(scenes['cloud'] < 20) & (scenes['mission'] == 'S2B')])

print(scenes[(scenes['cloud'] < 5) | (scenes['cloud'] > 60)])

# scenes[(scenes['cloud'] < 20) and (scenes['mission'] == 'S2B')]   # ValueError: truth value of a Series is ambiguous

print((scenes['cloud'] < 20).sum(), 'clear scenes')   # True counts as 1

## 5.3 New columns, dates and sorting

Arithmetic on columns works on **every row at once**, with no loop needed:

In [ ]:
scenes['ndvi'] = (scenes['nir'] - scenes['red']) / (scenes['nir'] + scenes['red'])

scenes['date'] = pd.to_datetime(scenes['date'])     # text -> real dates
scenes['month'] = scenes['date'].dt.month
scenes['doy'] = scenes['date'].dt.dayofyear

scenes.sort_values('ndvi', ascending=False)

You can also apply **your own function** to every value of a column with `.apply()`. For example, the Week 3 function that classifies NDVI:

In [ ]:
def ndvi_class(ndvi):
    """Classify an NDVI value into a broad (illustrative) cover class."""
    if ndvi < 0:
        return 'water'
    elif ndvi < 0.2:
        return 'bare/built'
    elif ndvi < 0.5:
        return 'sparse vegetation'
    return 'dense vegetation'

scenes['cover'] = scenes['ndvi'].apply(ndvi_class)
scenes[['date', 'cloud', 'ndvi', 'cover']]

#### Exercise 8

1. Add a column `usable` that is `True` when cloud cover is below 20 %.
2. Keep only the usable scenes, and sort them by **date**.
3. Add a column `days_since_previous` with the number of days between each usable scene and the one before. (Hint: `.diff()` on the date column, then `.dt.days`.)
4. Which month has the highest mean NDVI? (Hint: `groupby`, in the next section, or a filter per month.)

In [ ]:
scenes['usable'] = scenes['cloud'] < 20

usable = scenes[scenes['usable']].sort_values('date').copy()   # .copy() avoids a SettingWithCopy warning
usable['days_since_previous'] = usable['date'].diff().dt.days
print(usable[['date', 'mission', 'cloud', 'ndvi', 'days_since_previous']])

print(scenes.groupby('month')['ndvi'].mean().idxmax())

## 5.4 Grouping and summarising

`groupby()` follows the **split → apply → combine** pattern: split the rows into groups, compute a summary for each group, and combine the results into a new table.

In [ ]:
scenes.groupby('mission')['cloud'].mean()

In [ ]:
summary = scenes.groupby('month').agg(
    n_scenes=('date', 'count'),
    mean_cloud=('cloud', 'mean'),
    max_ndvi=('ndvi', 'max'),
).round(2)
summary

## 5.5 Saving and reloading

`df.to_csv(path, index=False)` saves a table. `index=False` stops pandas writing the row numbers as an extra column. Always **read the file back** once to check it.

In [ ]:
out_csv = data_dir / 'processed' / 'scenes_summary.csv'
summary.to_csv(out_csv)            # here the index (month) IS meaningful, so we keep it
check = pd.read_csv(out_csv)
check

---
# 6. Real data are messy: reading the Heathrow record

`pd.read_csv()` is the workhorse for reading tables. Its most useful arguments are:

| Argument | Use |
|---|---|
| `sep=','` | column separator; `sep=r'\s+'` means "one or more spaces or tabs" |
| `skiprows=n` | skip `n` header lines |
| `header=None`, `names=[...]` | the file has no usable header, so give the names yourself |
| `na_values=['---', -99.9]` | values that mean "missing" |
| `usecols=[...]` | read only some columns |
| `parse_dates=[...]` | convert columns to dates |
| `engine='python'` | a slower but more flexible parser |

Let's see what happens if we read the Heathrow file naively:

In [ ]:
try:
    naive = pd.read_csv(heathrow_file)
    print(naive.head())
except Exception as e:
    print(type(e).__name__, ':', e)

That is not a usable table. Looking at the file (section 4.2), we can list the problems:

1. **7 header lines** of text before the data start;
2. columns separated by **variable numbers of spaces**, not commas;
3. **missing values** written as `---`;
4. **flags stuck to numbers**: `*` = estimated, `#` = automatic sunshine sensor (e.g. `206.3#`);
5. an extra word **`Provisional`** on the most recent rows, so some rows have 8 fields instead of 7.

We deal with each one in turn, inside a **function**, so it can be reused (for any Met Office station file) and tested.

In [ ]:
def find_data_start(lines):
    """Return the index of the first line that starts with a 4-digit year.

    Parameters
    ----------
    lines : list of str
        Lines of a text file.

    Returns
    -------
    int
        Index of the first data line.
    """
    for i, line in enumerate(lines):
        parts = line.split()
        if len(parts) > 1 and len(parts[0]) == 4 and parts[0].isdigit():
            return i
    raise ValueError('no data line found')


def read_met_station(path):
    """Read a Met Office historic station data file into a clean DataFrame.

    Handles header lines, whitespace separation, '---' missing values,
    '*' (estimated) and '#' (sensor) flags and the 'Provisional' marker.

    Parameters
    ----------
    path : str or Path
        Path to the downloaded text file.

    Returns
    -------
    pandas.DataFrame
        Columns: date, year, month, tmax, tmin, af, rain, sun (numeric,
        NaN where missing) and provisional (bool).
    """
    path = Path(path)
    lines = path.read_text().splitlines()
    start = find_data_start(lines)

    names = ['year', 'month', 'tmax', 'tmin', 'af', 'rain', 'sun', 'status']
    df = pd.read_csv(path, sep=r'\s+', skiprows=start, header=None,
                     names=names, dtype=str, engine='python')

    for col in ['tmax', 'tmin', 'af', 'rain', 'sun']:
        cleaned = df[col].str.replace(r'[*#]', '', regex=True)       # remove flags
        df[col] = pd.to_numeric(cleaned, errors='coerce')           # '---' -> NaN

    df['year'] = df['year'].astype(int)
    df['month'] = df['month'].astype(int)
    df['provisional'] = df['status'] == 'Provisional'
    df['date'] = pd.to_datetime(dict(year=df['year'], month=df['month'], day=1))
    return df.drop(columns='status')[['date', 'year', 'month', 'tmax', 'tmin',
                                      'af', 'rain', 'sun', 'provisional']]

In [ ]:
heathrow = read_met_station(heathrow_file)
print(heathrow.shape)
heathrow.head()

In [ ]:
heathrow.tail()

In [ ]:
# Check: data types and missing values
heathrow.info()
print(heathrow.isna().sum())

In [ ]:
# Tests: the function must handle the tricky cases
test_file = data_dir / 'raw' / 'test_station.txt'
test_file.write_text(
    'Header line\n'
    '   yyyy  mm   tmax    tmin      af    rain     sun\n'
    '   1990   1    6.1*    1.2      ---    55.0    40.5\n'
    '   2026   9   23.6    13.8       0    26.2   147.8#  Provisional\n'
)
t = read_met_station(test_file)
assert len(t) == 2
assert t.loc[0, 'tmax'] == 6.1                 # '*' flag removed
assert pd.isna(t.loc[0, 'af'])                 # '---' became NaN
assert t.loc[1, 'sun'] == 147.8                # '#' flag removed
assert t['provisional'].tolist() == [False, True]
print('read_met_station tests passed')

## 6.1 Analysing the record

With a clean DataFrame, questions become one-liners:

In [ ]:
# Annual means (only years with all 12 months, to avoid biased partial years)
months_per_year = heathrow.groupby('year')['tmax'].count()
complete_years = months_per_year[months_per_year == 12].index

annual = (heathrow[heathrow['year'].isin(complete_years)]
          .groupby('year')
          .agg(tmax_mean=('tmax', 'mean'), tmin_mean=('tmin', 'mean'), rain_total=('rain', 'sum'))
          .round(2))
print(len(annual), 'complete years')
annual.tail(10)

In [ ]:
# The 5 hottest months on record (by mean daily maximum temperature)
heathrow.sort_values('tmax', ascending=False).head(5)[['date', 'tmax', 'rain', 'sun']]

> If you are using the offline excerpt, you will only see a few rows and probably no complete years. That is expected.

#### Exercise 9

Using the `heathrow` DataFrame:

1. Select **July** of every year, and print the 5 Julys with the highest `tmax`.
2. Compute the mean July `tmax` for **1961–1990** and for **1991–2020** (two standard climate-normal periods). How much warmer is the recent period?
3. Count the number of months per **decade** with `tmax` above 25 °C. (Hint: make a `decade` column with `(heathrow['year'] // 10) * 10`.)
4. Which **year** had the most air-frost days (`af`) in total?

In [ ]:
july = heathrow[heathrow['month'] == 7]
print(july.sort_values('tmax', ascending=False).head(5)[['year', 'tmax']])

normal_old = july[(july['year'] >= 1961) & (july['year'] <= 1990)]['tmax'].mean()
normal_new = july[(july['year'] >= 1991) & (july['year'] <= 2020)]['tmax'].mean()
print(f'July tmax 1961-1990: {normal_old:.2f} degC | 1991-2020: {normal_new:.2f} degC | difference: {normal_new - normal_old:+.2f} degC')

heathrow['decade'] = (heathrow['year'] // 10) * 10
print(heathrow[heathrow['tmax'] > 25].groupby('decade').size())

af_year = heathrow.groupby('year')['af'].sum(min_count=1)
print('most air-frost days:', af_year.idxmax(), af_year.max())

#### Exercise 10: The Oceanic Niño Index

1. Read `data/raw/oni.ascii.txt` (from Exercise 6) with `pd.read_csv(..., sep=r'\s+')`. Inspect it with `head()`, `info()` and `describe()`.
2. ENSO phases are usually defined as **El Niño** for `ANOM >= 0.5` and **La Niña** for `ANOM <= -0.5`. Add a column `phase` with the values `'El Nino'`, `'La Nina'` or `'Neutral'` (hint: a function and `.apply()`, as for NDVI).
3. For each year from **2015 onwards**, count the number of El Niño seasons (hint: filter, then `groupby('YR').size()`).
4. Find the season with the strongest El Niño on record.

In [ ]:
oni = pd.read_csv(oni_file, sep=r'\s+')
print(oni.head())
oni.info()
print(oni.describe())

def enso_phase(anom, threshold=0.5):
    """Classify an ONI anomaly as El Nino, La Nina or Neutral."""
    if anom >= threshold:
        return 'El Nino'
    elif anom <= -threshold:
        return 'La Nina'
    return 'Neutral'

oni['phase'] = oni['ANOM'].apply(enso_phase)

recent = oni[(oni['YR'] >= 2015) & (oni['phase'] == 'El Nino')]
print(recent.groupby('YR').size())

print(oni.loc[oni['ANOM'].idxmax()])

---
# 7. Formative exercise: a reusable data pipeline

Put the week together in the way Part B expects: **small functions with docstrings, chained into a workflow**, and outputs saved in usable formats.

Write a function `station_summary(url, station_name, out_dir)` that:

1. **downloads** the station file into `out_dir / 'raw'` with `download_file()` (falling back to the excerpt is fine for Heathrow);
2. **reads and cleans** it with `read_met_station()`;
3. computes **annual** mean `tmax` and `tmin` and total `rain`, for complete years only;
4. **saves** the annual table as `out_dir / 'processed' / f'{station_name}_annual.csv'`;
5. saves a **provenance JSON** file next to it (URL, download time, number of years, first and last year);
6. **returns** the annual DataFrame.

Test it on Heathrow. If you have internet access, also run it on another Met Office station by changing the URL: the file names follow the same pattern, e.g. `.../stationdata/oxforddata.txt` or `.../stationdata/durhamdata.txt`.

Then reload the saved CSV and use `assert` to check that it has the same number of rows as the returned DataFrame.

In [ ]:
def annual_summary(df):
    """Annual mean tmax/tmin and total rain for years with 12 months of tmax data."""
    n = df.groupby('year')['tmax'].count()
    complete = n[n == 12].index
    return (df[df['year'].isin(complete)]
            .groupby('year')
            .agg(tmax_mean=('tmax', 'mean'), tmin_mean=('tmin', 'mean'), rain_total=('rain', 'sum'))
            .round(2))


def station_summary(url, station_name, out_dir):
    """Download, clean, summarise and save a Met Office station record.

    Parameters
    ----------
    url : str
        URL of the Met Office station data text file.
    station_name : str
        Short name used in output file names, e.g. 'heathrow'.
    out_dir : str or Path
        Base output folder; 'raw' and 'processed' sub-folders are used.

    Returns
    -------
    pandas.DataFrame
        Annual summary indexed by year.
    """
    out_dir = Path(out_dir)
    raw_file = out_dir / 'raw' / f'{station_name}data.txt'
    try:
        download_file(url, raw_file)
    except Exception as e:
        if station_name == 'heathrow':
            print('download failed, using excerpt:', e)
            raw_file.parent.mkdir(parents=True, exist_ok=True)
            raw_file.write_text(HEATHROW_EXCERPT)
        else:
            raise

    df = read_met_station(raw_file)
    annual = annual_summary(df)

    processed = out_dir / 'processed'
    processed.mkdir(parents=True, exist_ok=True)
    csv_path = processed / f'{station_name}_annual.csv'
    annual.to_csv(csv_path)

    prov = {
        'station': station_name,
        'url': url,
        'downloaded': datetime.fromtimestamp(raw_file.stat().st_mtime).isoformat(timespec='seconds'),
        'n_complete_years': len(annual),
        'first_year': int(annual.index.min()) if len(annual) else None,
        'last_year': int(annual.index.max()) if len(annual) else None,
    }
    (processed / f'{station_name}_provenance.json').write_text(json.dumps(prov, indent=2))
    return annual


heathrow_annual = station_summary(HEATHROW_URL, 'heathrow', data_dir)
print(heathrow_annual.tail())

reloaded = pd.read_csv(data_dir / 'processed' / 'heathrow_annual.csv')
assert len(reloaded) == len(heathrow_annual)
print('saved and reloaded OK')

---
# Summary

| Topic | Key points |
|---|---|
| Paths | `Path('a') / 'b.txt'`; `.name`, `.stem`, `.suffix`, `.parent`, `.exists()`, `.mkdir(parents=True, exist_ok=True)` |
| Text files | `read_text()` / `write_text()`; `with open(path, 'w' / 'a' / 'r') as f:` |
| Finding files | `glob('*.tif')`, `rglob()`; **sort** the results; zero-pad names |
| Copy / delete | `shutil.copy`, `rename`, `unlink(missing_ok=True)`, `shutil.rmtree` (careful!) |
| Dates | `datetime.strptime` / `strftime`, `timedelta`, `%j` for day of year |
| JSON / YAML | `json.dumps` / `json.loads`; `yaml.safe_load` / `yaml.safe_dump`; config files |
| URLs | `requests.get(url, timeout=...)`, `raise_for_status()`, download once and reuse, record provenance |
| Errors | `try:` ... `except Exception as e:` |
| Credentials | never in a notebook: Colab Secrets, `getpass`, login helpers |
| pandas: inspect | `head`, `info`, `describe`, `shape`, `value_counts`, `isna().sum()` |
| pandas: select | `df['c']`, `df[['a', 'b']]`, `df[cond]`; combine conditions with `&` (and), the pipe (or) and `~` (not), each in brackets; `loc`, `iloc` |
| pandas: transform | new columns from arithmetic; `pd.to_datetime`, `.dt.month`; `.apply(func)`; `sort_values` |
| pandas: summarise | `groupby(...).agg(name=('col', 'func'))` |
| pandas: read / write | `read_csv(sep=r'\s+', skiprows=, names=, na_values=)`; `to_csv(index=False)` |

**Next week (Week 5):** plotting with `matplotlib`, starting with the Heathrow and ONI time series you prepared today, so keep the `data/` folder (save it to Drive!).

---
# Resources

## Cheat sheets and references

| Resource | What it covers |
|---|---|
| [pandas cheat sheet (official PDF)](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) | Two pages: selecting, filtering, new columns, grouping, reshaping. **Print this one.** |
| [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | The official quick tour |
| [Python Crash Course cheat sheets](https://ehmatthes.github.io/pcc_3e/cheat_sheets/) | This week: ***Files and Exceptions*** |
| [`pathlib` documentation](https://docs.python.org/3/library/pathlib.html) | All `Path` methods, with a table mapping old `os.path` functions to `pathlib` |
| [`datetime` format codes](https://docs.python.org/3/library/datetime.html#format-codes) | Every `%` code for `strptime` and `strftime` |
| [QuickRef Python 3](https://quickref.me/python) | General syntax reminder, including file handling |

## Practise and check your code

| Site | Good for this week | How it checks |
|---|---|---|
| [Kaggle Learn: Pandas](https://www.kaggle.com/learn/pandas) | A short course of hands-on pandas exercises in the browser: indexing, filtering, grouping, data types, missing values | Each exercise has a `check()` that tells you if your answer is right, plus hints |
| [W3Schools pandas tutorial](https://www.w3schools.com/python/pandas/default.asp) | Simple examples on reading CSVs, cleaning data and `describe()` | Try-it-yourself editor; exercises with right/wrong feedback |
| [Python Tutor](https://pythontutor.com/) | Seeing what `with open(...)` and loops over file lines do (pandas itself is not supported) | Step-by-step visualisation |
| [Exercism Python track](https://exercism.org/tracks/python) | Exercises on string processing and dictionaries, which are useful for parsing files | Automated tests |
| [Codewars](https://www.codewars.com/) | Search kata tagged *Data Science* or *Strings* | Automated tests |
| [W3Schools Python exercises](https://www.w3schools.com/python/python_exercises.asp) | *File handling*, *try/except* and *dates* drills | Instant right/wrong |

## Python for environmental and Earth data

- [Earth Data Science textbook (Earth Lab)](https://www.earthdatascience.org/courses/intro-to-earth-data-science/): see the chapters on working with files and directories and on tabular data with pandas.
- [Software Carpentry: Plotting and Programming in Python](https://swcarpentry.github.io/python-novice-gapminder/): a well-tested beginner lesson that uses pandas on real data (and leads into next week's plotting).

> **Tip:** when you meet a new data file, the most valuable habit is the one from section 6: **look at the raw text first**, list what is awkward about it, then write a reading function that handles each problem and test it.